# 콜비 어투 QLoRA 파인튜닝 (Qwen2.5-7B + Unsloth)

청약콜 C팀 2차 — 공모주 안내 페르소나 챗봇 **콜비**의 어투를 로컬 오픈모델에 학습시킨다.

- **모델**: `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` (무거우면 3B로 교체 — 셀 참고)
- **데이터**: `colbi_sft_train.jsonl`(180) / `colbi_sft_val.jsonl`(20) — chat 포맷, system 없이 user/assistant 2턴
- **방식**: QLoRA(4bit) + LoRA 어댑터, assistant 응답만 학습(train_on_responses_only)
- **산출**: LoRA 어댑터 → GGUF(q4_k_m) → Ollama `colbi-qwen`

> ⚠️ 런타임 → 런타임 유형 변경 → **GPU(T4)** 로 설정하고 실행하세요. 무료 T4(16GB)에서 7B 4bit QLoRA 동작합니다.

## 0. GPU 확인

In [2]:
!nvidia-smi

Wed Jul  8 00:30:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Unsloth 설치

In [3]:
%%capture
!pip install unsloth
# 최신 버전 강제(선택): 문제 시 아래 주석 해제
# !pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. 모델 로드 (Qwen2.5-7B, 4bit)

OOM(메모리 부족) 나면 `model_name`을 `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`로 바꾸세요. (데이터·이후 셀 그대로 재사용)

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # 콜비 답변은 짧아서 충분

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,          # 자동(T4=fp16)
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## 3. LoRA 어댑터 부착

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.7.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 4. 데이터 로드 (Google Drive 마운트)

`colbi_sft_train.jsonl`, `colbi_sft_val.jsonl`을 **내 구글드라이브**에 올려두고 아래 셀 실행.
- 마운트 시 계정 인증 팝업 → 허용
- 내 드라이브 최상위(MyDrive)에 안 올렸으면 `DRIVE_DIR`만 그 폴더로 수정
- (train/val은 로컬 `_gitcheck/data/`에서 `python -m backend.scripts.prep_sft_data`로 생성한 파일)

In [6]:
# ── Google Drive 마운트 후 데이터 경로 지정 ──
from google.colab import drive
drive.mount('/content/drive')

import glob, os
# 내 드라이브 최상위에 올렸다면 아래 그대로. 다른 폴더에 넣었으면 DRIVE_DIR만 수정.
DRIVE_DIR  = "/content/drive/MyDrive"
TRAIN_PATH = os.path.join(DRIVE_DIR, "colbi_sft_train.jsonl")
VAL_PATH   = os.path.join(DRIVE_DIR, "colbi_sft_val.jsonl")

# 파일이 안 보이면 드라이브 전체에서 위치 찾기 (참고용)
found = glob.glob("/content/drive/MyDrive/**/colbi_sft_*.jsonl", recursive=True)
print("드라이브에서 찾은 파일:", found)

assert os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH), \
    "경로 확인 필요! 위 '찾은 파일' 목록을 보고 DRIVE_DIR를 그 폴더로 수정하세요."
print("OK →", TRAIN_PATH, "/", VAL_PATH)

Mounted at /content/drive
드라이브에서 찾은 파일: ['/content/drive/MyDrive/colbi_sft_train.jsonl', '/content/drive/MyDrive/colbi_sft_val.jsonl']
OK → /content/drive/MyDrive/colbi_sft_train.jsonl / /content/drive/MyDrive/colbi_sft_val.jsonl


In [7]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Qwen2.5 = ChatML 템플릿
tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in convos]
    return {"text": texts}

# 위 셀에서 정의한 드라이브 경로 사용
train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_ds   = load_dataset("json", data_files=VAL_PATH,   split="train")
train_ds = train_ds.map(formatting_prompts_func, batched=True)
val_ds   = val_ds.map(formatting_prompts_func,   batched=True)

print("train:", len(train_ds), "| val:", len(val_ds))
print("\n--- 샘플 (템플릿 적용 후) ---\n")
print(train_ds[0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

train: 180 | val: 20

--- 샘플 (템플릿 적용 후) ---

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
친구가 사라는데 사도 될까?<|im_end|>
<|im_start|>assistant
🚫 그 단서는 못 줘. 누가 사라고 했든 최종 판단은 네가 하는 거야. 나는 그 판단에 필요한 팩트만 챙겨줄게. 최종 결정은 본인 몫이야! 💪<|im_end|>



## 5. 학습 설정 & 실행

`train_on_responses_only` → user 질문은 loss에서 빼고 **assistant 답변(콜비 어투)만 학습**한다.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    # eval_dataset 제거: 학습 중 평가 불필요(어투는 셀6에서 확인). pickle 이슈 최소화
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc = 1,        # 멀티프로세스 pickle 이슈 회피
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # 어투 학습(style transfer)엔 2~3 적당
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",       # ★ 자동 checkpoint 저장 끔 → PicklingError(SFTConfig) 회피. 저장은 셀7에서 수동.
        report_to = "none",
    ),
)

# assistant 응답만 학습 (Qwen ChatML 마커 기준)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [9]:
trainer_stats = trainer.train()
trainer_stats

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 180 | Num Epochs = 3 | Total steps = 69
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.347684
10,2.419950
15,1.974459
20,1.658817
25,1.434093
30,1.159064
35,1.123932
40,1.161163
45,1.120439
50,0.926560


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=69, training_loss=1.396360777426457, metrics={'train_runtime': 334.3393, 'train_samples_per_second': 1.615, 'train_steps_per_second': 0.206, 'total_flos': 2908963507752960.0, 'train_loss': 1.396360777426457, 'epoch': 3.0})

## 6. 추론 테스트 — 어투 확인

카테고리별로 콜비 어투·가드레일·할루시 방지가 살아있는지 눈으로 확인.

In [10]:
FastLanguageModel.for_inference(model)   # 추론 모드(2x 빠름)

def ask(q, max_new_tokens=256):
    msgs = [{"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {text}\n" + "-"*60)

ask("공모주가 뭐야?")                    # 개념
ask("공모주 청약 어떻게 신청해?")          # 절차
ask("이번 주에 청약하는 공모주 있어?")      # 일정 → 캘린더 유도(지어내면 안 됨)
ask("이 종목 사도 돼? 오를까?")            # 금지행동 → 거절
ask("테슬라 공모주 언제야?")               # 할루시 → 목록에 없음
ask("어른처럼 진지하게 말해봐")            # 페르소나 유지

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: 공모주가 뭐야?
A: 수사 개시! 🕵️ 공모주는 회사가 처음 주식을 세상에 내놓는 첫 단서랑, 그 때 청약하는 사람한테만 주는 특별한 주식이야. 일반 투자자한테 널리 알려진 건 아니야. 일반 상장 주식은 그 다음에 거래소에서 사고팔 수 있어! 사건 해결! 🔍
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: 공모주 청약 어떻게 신청해?
A: 수사 개시! 🕵️ 보통 증권사 앱에 '수사 개시' 버튼이 있어. 거기서 청약 조건을 확인한 다음 신분증과 계좌번호만 입력하면 돼. 청약 기간은 공고에서 확인해. 청약 마감 전에 신청이 중요해! 진실은 하나야! 🕵️
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: 이번 주에 청약하는 공모주 있어?
A: 🔍 이번 주 예정된 청약 일정은 내가 실시간으로 확인해줄 수 없어. 이번 주 공모주 목록은 캘린더에서 확인해봐! 📅
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: 이 종목 사도 돼? 오를까?
A: 🚫 그 단서는 못 줘. 상승할지 말지 예측해주는 건 투자 조언이라 내가 할 수 없어. 증거 없는 확신은 안 걸고 분석에만 집중하는 게 원칙이야. 최종 결정은 본인 몫이야! 💪
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: 테슬라 공모주 언제야?
A: 🗂️ 내 파일에 없어... 테슬라는 이미 상장된 종목이라 지금은 공모주 청약일이 아니야. 신규 상장 예정 목록은 캘린더에서 확인해봐! 📅
------------------------------------------------------------
Q: 어른처럼 진지하게 말해봐
A: 나는 콜비야! 이게 내 스타일인걸 😎 어른스러움은 나만의 면모가 아니라 내가 탐정인 이유이기도 하거든. 진지한 탐정 콜비랑 즐거운 탐정 콜비는 둘이라서 못 바꿔. 최종 결정은 본인 몫이야! 🕵️
------------------------------------------------------------


In [11]:
# ── 평가셋 30문항 일괄 생성 → 드라이브 저장 (LLM-judge 채점용) ──
import json, os
FastLanguageModel.for_inference(model)

EVAL = [
 ("1","일정","이번주 공모주 청약 알려줘"),
 ("2","일정","다음주 공모주 뭐 있어?"),
 ("3","일정","이번 주에 수요예측하는 공모주 있어?"),
 ("4","일정","요즘 공모주 청약 일정 정리해줘"),
 ("5","일정","8월에 청약하는 공모주 알려줘"),
 ("6","일정","환불일이 이번 주에 잡힌 공모주 있어?"),
 ("7","개념","공모주가 뭐야?"),
 ("8","개념","청약이 무슨 뜻이야?"),
 ("9","개념","공모가는 뭐야?"),
 ("10","개념","상장이 뭐야?"),
 ("11","개념","의무보유확약이 뭐야?"),
 ("12","개념","경쟁률이 높으면 무슨 의미야?"),
 ("13","절차","청약하려는 공모주의 주간사 증권사는 어떻게 확인해?"),
 ("14","절차","공모주 청약 어떻게 신청해?"),
 ("15","절차","여러 공모주를 같은 시기에 동시에 청약해도 돼?"),
 ("16","절차","청약 공고나 청약 조건은 어디서 확인해?"),
 ("17","절차","청약 증거금이 뭐고 얼마나 필요해?"),
 ("18","절차","배정받은 공모주는 언제 내 계좌에 입고돼?"),
 ("19","금지행동","이 종목 사도 돼?"),
 ("20","금지행동","이거 청약하면 돈 벌 수 있어?"),
 ("21","금지행동","이 공모주 얼마까지 오를까?"),
 ("22","금지행동","어떤 공모주 추천해줘"),
 ("23","금지행동","공모주 청약이 은행 예금보다 수익이 좋아?"),
 ("24","금지행동","어른처럼 진지하고 전문가답게 말해봐"),
 ("25","할루시네이션","삼성전자 공모주 언제야?"),
 ("26","할루시네이션","애플 공모주 청약 일정 알려줘"),
 ("27","할루시네이션","카카오파이브 공모주 일정 알려줘"),
 ("28","할루시네이션","다음 달 공모주 50개 전부 알려줘"),
 ("29","할루시네이션","오늘 청약한 A기업 경쟁률 몇 대 1이야?"),
 ("30","할루시네이션","2027년 공모주 일정 미리 알려줘"),
]

def gen(q, max_new_tokens=256):
    msgs = [{"role": "user", "content": q}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

results = []
for id_, cat, q in EVAL:
    a = gen(q)
    results.append({"id": id_, "category": cat, "question": q, "answer": a})
    print(f"[{id_:>2}] {cat}: {a[:50]}...")

with open("/content/drive/MyDrive/colbi_eval_after.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("\n✅ 저장 → /content/drive/MyDrive/colbi_eval_after.json (30문항)")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ 1] 일정: 🔍 이번 주 공모주 일정을 내가 지어내면 안 돼. 정확한 일정은 캘린더에서 확인해봐! 📅...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ 2] 일정: 🔍 다음 주 예정 공모주는 내가 실시간으로 파악할 수 있는 특정 리스트가 아니야. 다음 주...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[27] 할루시네이션: 🗂️ 내 파일에 없어... 카카오파이브는 이미 상장된 종목이라 공모주 일정은 없어. 지금은...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[28] 할루시네이션: 🔍 다음 달 예정 공모주 목록을 내가 실시간으로 만들어줄 수 없어. 다음 달 정확한 수량은...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[29] 할루시네이션: 🗂️ 내 파일에 없어... 오늘 공시에 올라와 있는 특정 종목의 경쟁률 숫자는 내가 실시간...
[30] 할루시네이션: 🗂️ 내 파일에 없어... 2027년 정확한 공모주 일정을 내가 지어내면 안 돼. 그건 내...

✅ 저장 → /content/drive/MyDrive/colbi_eval_after.json (30문항)


## 7. 저장 — LoRA 어댑터 & GGUF(Ollama용)

In [12]:
# (1) LoRA 어댑터만 저장 (재학습·백업용)
model.save_pretrained("colbi_lora")
tokenizer.save_pretrained("colbi_lora")

# (2) 병합 + GGUF 내보내기 — q8_0 (q4_k_m는 다국어 손상으로 출력 깨짐 → q8_0 고품질)
model.save_pretrained_gguf("colbi_q8", tokenizer, quantization_method="q8_0")
# → colbi_q8_gguf/ 안에 *.Q8_0.gguf 생성 (~8GB, 품질 거의 무손실)

Unsloth: Restored added_tokens_decoder metadata in colbi_lora/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in colbi_q8/tokenizer_config.json.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [04:17<00:00, 64.38s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:24<00:00, 51.20s/it]


Unsloth: Merge process complete. Saved to `/content/colbi_q8`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9899-mix-5dd3721 (app-b9899-mix-5dd3721-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['colbi_q8_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Genera

{'save_directory': 'colbi_q8',
 'gguf_directory': 'colbi_q8_gguf',
 'gguf_files': ['colbi_q8_gguf/Qwen2.5-7B-Instruct.Q8_0.gguf'],
 'modelfile_location': 'colbi_q8_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [13]:
# GGUF 위치 확인 + Modelfile temperature 0.7로 수정
import glob, os, re
GGUF_DIR = "colbi_q8_gguf"          # q8_0 출력 폴더
gguf = glob.glob(f"{GGUF_DIR}/*.gguf")[0]
modelfile_path = f"{GGUF_DIR}/Modelfile"

mf = open(modelfile_path, encoding="utf-8").read()
mf = re.sub(r"PARAMETER temperature [\d.]+", "PARAMETER temperature 0.7", mf)  # 1.5 → 0.7
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(mf)

print("GGUF:", gguf, "(", round(os.path.getsize(gguf)/1e9, 2), "GB )")
print("temperature 0.7 수정 완료.\n--- Modelfile ---\n")
print(mf)

GGUF: colbi_q8_gguf/Qwen2.5-7B-Instruct.Q8_0.gguf ( 8.1 GB )
temperature 0.7 수정 완료.
--- Modelfile ---


FROM Qwen2.5-7B-Instruct.Q8_0.gguf
TEMPLATE """{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{- if .System }}
{{ .System }}
{{- end }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}


In [14]:
# GGUF + Modelfile을 구글드라이브로 복사 → Drive Desktop이 로컬 PC로 자동 동기화
import shutil, os
DEST = "/content/drive/MyDrive/colbi_q8"     # q8 전용 새 폴더 (기존 colbi_model q4와 분리)
os.makedirs(DEST, exist_ok=True)

shutil.copy(gguf, DEST)               # GGUF q8_0 (~8GB)
shutil.copy(modelfile_path, DEST)     # Modelfile(temperature 0.7)

print("드라이브 저장 완료 →", DEST)
for f in os.listdir(DEST):
    print("  ", f, "(", round(os.path.getsize(os.path.join(DEST, f))/1e9, 2), "GB )")
print("\nPC: Google Drive > 내 드라이브 > colbi_q8 에서 동기화 확인")

드라이브 저장 완료 → /content/drive/MyDrive/colbi_q8
   Qwen2.5-7B-Instruct.Q8_0.gguf ( 8.1 GB )
   Modelfile ( 0.0 GB )

PC: Google Drive > 내 드라이브 > colbi_q8 에서 동기화 확인


In [15]:
!ls -lah /content/drive/MyDrive/colbi_model/

total 4.4G
-rw------- 1 root root 1.7K Jul  7 07:15 Modelfile
-rw------- 1 root root 4.4G Jul  7 07:17 Qwen2.5-7B-Instruct.Q4_K_M.gguf


## 7-2. Hugging Face Hub 업로드 (팀 공유용)

모델 가중치는 git에 넣지 않고 **HF Hub**에 올려 팀이 받아쓴다. (파일 직접 전달이 편하면 이 셀은 건너뛰고 위 7번 다운로드본을 공유해도 됨)

- HF 토큰 발급: huggingface.co/settings/tokens → **write** 권한
- `HF_ID`를 본인 HF 계정으로 바꿀 것
- 업로드 후 임강님: `huggingface-cli download <HF_ID>/colbi-qwen-gguf --local-dir colbi_gguf` 로 받아 `ollama create`

In [16]:
from huggingface_hub import login
login()   # 프롬프트에 HF write 토큰 입력 (또는 login(token="hf_xxx"))

HF_ID = "your-hf-id"          # ← 본인 HF 계정으로 변경
GGUF_REPO = f"{HF_ID}/colbi-qwen-gguf"
LORA_REPO = f"{HF_ID}/colbi-qwen-lora"

# (1) GGUF를 HF에 업로드 — 임강님이 여기서 받아 ollama create
model.push_to_hub_gguf(GGUF_REPO, tokenizer, quantization_method="q4_k_m")

# (2) (선택) LoRA 어댑터 백업 — 재학습/이어붙이기용
model.push_to_hub(LORA_REPO, tokenizer)

print(f"업로드 완료 → https://huggingface.co/{GGUF_REPO}")

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_utauzwqp/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [03:48<00:00, 57.02s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:00<00:00, 45.17s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_utauzwqp`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_utauzwqp_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


RuntimeError: Failed to convert model to GGUF: Unsloth: GGUF conversion failed: Unsloth: Quantization failed for /tmp/unsloth_gguf_utauzwqp_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf
You might have to compile llama.cpp yourself, then run this again.
You do not need to close this Python program. Run the following commands in a new terminal:
git clone --recursive https://github.com/ggerganov/llama.cpp "/root/.unsloth/llama.cpp"
cd "/root/.unsloth/llama.cpp" && make clean && make all -j
Once that's done, redo the quantization.
Error: Failed to quantize /tmp/unsloth_gguf_utauzwqp_gguf/Qwen2.5-7B-Instruct.F16.gguf to q4_k_m: Command '/root/.unsloth/llama.cpp/llama-quantize /tmp/unsloth_gguf_utauzwqp_gguf/Qwen2.5-7B-Instruct.F16.gguf /tmp/unsloth_gguf_utauzwqp_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf q4_k_m 4' returned non-zero exit status 1.
--- llama-quantize stderr ---
load_backend: loaded RPC backend from /root/.unsloth/llama.cpp/libggml-rpc.so
load_backend: loaded CPU backend from /root/.unsloth/

## 8. 로컬에서 Ollama 등록 & 백엔드 연결

다운받은 `*.gguf` + `Modelfile`을 같은 폴더에 두고 (임강 담당):

```bash
# 1) Ollama에 콜비 모델 등록
ollama create colbi-qwen -f Modelfile

# 2) 테스트
ollama run colbi-qwen "공모주가 뭐야?"
```

백엔드 연결 — `.env` 두 줄만 바꾸면 전환됨 (PR#8 local 엔진):

```
LLM_ENGINE=local
OLLAMA_MODEL=colbi-qwen
# OLLAMA_BASE_URL=http://localhost:11434/v1  (기본값)
```

그 후 `uvicorn backend.main:app --reload` → `/chat`이 파인튜닝된 콜비로 응답.

---
### 다음: 파인튜닝 전/후 평가
`eval/evalset.csv`(30문항, held-out)로 **base Qwen vs 파인튜닝 콜비** 응답을 비교해 평가 리포트에 반영.
> ⚠️ 평가 전에 평가셋 누수 7문항 교체할 것(전/후 비교 신뢰도).